## Part 2: Prepare ELSET Data
#### 1. Acquire the latest General Perturbation (gp) Data in form of Orbit Mean-Element Messages (OMMs) from [space-track.org](https://www.space-track.org/)

In [38]:
# Reauthenticate with Space-Track.org to avoid having to run first cell again

# Space-Track.org API authentication 
from spacetrack import SpaceTrackClient
import pandas as pd

# read Space-Track.org credentials from external file (username in line 1, password in line 2)
with open('../DS_SpaceTrackCredentials.txt', 'r') as file:
    credentials = file.read().splitlines()

username = credentials[0]
password = credentials[1]

# create a SpaceTrackClient instance
st = SpaceTrackClient(username, password) 

try:
    # Try fetching a single SATCAT entry to verify authentication
    test_data = st.gp(limit=1, format='json')
    print("Successfully authenticated with Space-Track.org!")
except Exception as e:
    print(f"Failed to authenticate with Space-Track.org: {e}")

Successfully authenticated with Space-Track.org!


In [39]:
# Read the NORAD IDs from the file
norad_ids = pd.read_csv("../Data/DS_NoradIDs.csv")
print(f"Objects: {len(norad_ids)}")

Objects: 25745


In [40]:
# Split the IDs into chunks of 500 to avoid violating the API query restrictions
norad_ids_chunks = []
chunk_size = 500
for i in range(0, len(norad_ids), chunk_size):
    chunk = norad_ids[i:i + chunk_size]
    norad_ids_chunks.append(chunk)

# show number and size of chunks
print(f"Number of chunks: {len(norad_ids_chunks)}")
print(f"Chunk size: {len(norad_ids_chunks[0])}; last chunk: {len(norad_ids_chunks[-1])}")

Number of chunks: 52
Chunk size: 500; last chunk: 245


In [41]:
# Convert each chunk to a string to avoid violating the API query limit
chunk_strings=[]
for chunk in norad_ids_chunks:
    if not chunk.empty:
        chunk_string = ",".join(chunk["NORAD_CAT_ID"].astype(str))
        chunk_strings.append(chunk_string)


In [42]:
# Retreive OMM data for each chunk
import time # Import the time module to add delays between API requests
import json # Import the json module to parse the API response

omm_data_chunks = []
omm_data_received = 0

for i, chunk_string in enumerate(chunk_strings):
    try:
        # retreive omm data
        omm_data = st.gp(norad_cat_id=chunk_string, format='json')
        omm_data_chunks.append(omm_data)

        # Parse the json response to count the elements
        omm_data_json = json.loads(omm_data)
        json_element_count = len(omm_data_json)

        print(f"Retrieved OMM data for chunk {i+1} with {len(chunk_string.split(','))} IDs requested and {json_element_count} elements returned")
        omm_data_received += json_element_count
    except Exception as e:
        print(f"Error retrieving OMM data for chunk {i+1}: {e}")

    # Add a delay of 5 seconds between API requests to avoid rate limiting
    if i < len(chunk_strings) - 1:
        print("Waiting for 5 seconds before the next request...")
        time.sleep(5)

print(f"Total OMMs requested: {len(norad_ids)}")
print(f"Total OMM data elements received: {omm_data_received}")
print("Missing: " + str(len(norad_ids) - omm_data_received))

Retrieved OMM data for chunk 1 with 500 IDs requested and 474 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 2 with 500 IDs requested and 368 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 3 with 500 IDs requested and 409 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 4 with 500 IDs requested and 480 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 5 with 500 IDs requested and 490 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 6 with 500 IDs requested and 402 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 7 with 500 IDs requested and 427 elements returned
Waiting for 5 seconds before the next request...
Retrieved OMM data for chunk 8 with 500 IDs requested and 255 elements returned
Waiting for 5 sec

In [43]:
# Save the OMM data chunks to a dataframe
dataframes = [pd.DataFrame(json.loads(chunk)) for chunk in omm_data_chunks]
OMMs_df = pd.concat(dataframes, ignore_index=True)

# show all columns of the df
pd.set_option('display.max_columns', None)
OMMs_df

,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,ORIGINATOR,OBJECT_NAME,OBJECT_ID,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,COUNTRY_CODE,LAUNCH_DATE,SITE,DECAY_DATE,FILE,GP_ID,TLE_LINE0,TLE_LINE1,TLE_LINE2
0,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:26,18 SPCS,THOR ABLESTAR R/B,1960-003A,EARTH,TEME,UTC,SGP4,1961-08-16T14:24:53.774496,16.29358349,0.00000000,51.2500,356.4892,331.9473,28.0526,0,U,30,999,7590,0.00000000000000,0.02125614,0.0000000000000,6572.405,88.378,194.270,194.270,ROCKET BODY,None,None,None,None,1961-08-18,33907,8672722,0 THOR ABLESTAR R/B,1 00030U 60003 A 61228.60062239 .02125614 +0...,2 00030 051.2500 356.4892 0000000 331.9473 028...
1,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,THOR ABLESTAR DEB,1960-003C,EARTH,TEME,UTC,SGP4,1960-10-06T07:59:06.496512,15.38343354,0.02427570,51.2879,136.0668,196.7800,163.6100,0,U,33,999,2700,0.00000000000000,0.00001932,0.0000000000000,6829.149,93.607,616.796,285.231,DEBRIS,None,None,None,None,1960-07-17,33906,8671947,0 THOR ABLESTAR DEB,1 00033U 60003 C 60280.33271408 .00001932 +0...,2 00033 051.2879 136.0668 0242757 196.7800 163...
2,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,SL-3 R/B,1960-005B,EARTH,TEME,UTC,SGP4,1960-07-16T04:00:11.150208,16.27523673,0.00000000,64.8899,44.6976,36.3819,323.6180,0,U,35,999,990,0.00000000000000,0.03271057,0.0000000000000,6577.344,88.477,199.209,199.209,ROCKET BODY,None,None,None,None,1960-07-17,33906,8671988,0 SL-3 R/B,1 00035U 60005 B 60198.16679572 .03271057 +0...,2 00035 064.8899 044.6976 0000000 036.3819 323...
3,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:26,18 SPCS,SPUTNIK 4 DEB,1960-005D,EARTH,TEME,UTC,SGP4,1961-03-15T15:48:42.802271,15.49800338,0.01984030,64.8899,343.2411,333.4201,27.0840,0,U,37,999,4690,0.00000000000000,0.00039007,0.0000000000000,6795.450,92.915,552.139,282.492,DEBRIS,None,None,None,None,1961-06-30,33907,8672884,0 SPUTNIK 4 DEB,1 00037U 60005 D 61074.65882873 .00039007 +0...,2 00037 064.8899 343.2411 0198403 333.4201 027...
4,3.0,GENERATED VIA SPACE-TRACK.ORG API,2004-08-15T23:35:23,18 SPCS,SPUTNIK 4 DEB,1960-005E,EARTH,TEME,UTC,SGP4,1960-08-19T16:34:33.722976,15.88909013,0.00342130,64.8899,302.1218,49.4823,310.6691,0,U,38,999,1500,0.00000000000000,0.00610805,0.0000000000000,6683.481,90.628,328.212,282.480,DEBRIS,None,None,None,None,1960-08-20,33906,8672029,0 SPUTNIK 4 DEB,1 00038U 60005 E 60232.69066809 .00610805 +0...,2 00038 064.8899 302.1218 0034213 049.4823 310...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24643,3.0,GENERATED VIA SPACE-TRACK.ORG API,2024-12-04T19:54:48,18 SPCS,CZ-2F DEB,2024-194B,EARTH,TEME,UTC,SGP4,2024-11-22T23:07:29.851680,15.68300172,0.02578460,41.3237,193.5705,294.4302,62.9836,0,U,61684,999,378,0.00048903000000,0.00465987,0.0000065563000,6741.901,91.819,537.603,189.929,DEBRIS,LARGE,PRC,2024-10-29,JSC,2024-12-21,4571547,274742509,0 CZ-2F DEB,1 61684U 24194B 24327.96353995 .00465987 6...,2 61684 41.3237 193.5705 0257846 294.4302 62...
24644,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-02-16T08:34:50,18 SPCS,CZ-6A DEB,2024-140ADL,EARTH,TEME,UTC,SGP4,2025-02-15T18:24:08.792352,15.92225175,0.00210800,89.2435,333.2672,98.2724,262.0952,0,U,61844,999,1983,0.04150000000000,0.09622810,0.2057300000000,6674.194,90.439,310.129,281.990,DEBRIS,SMALL,PRC,2024-08-06,TSC,2025-02-21,4642461,280861418,0 CZ-6A DEB,1 61844U 24140ADL 25046.76676843 .09622810 2...,2 61844 89.2435 333.2672 0021080 98.2724 262...
24645,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-02-19T02:46:15,18 SPCS,FALCON 9 R/B,2025-022Y,EARTH,TEME,UTC,SGP4,2025-02-19T02:13:45.786720,16.47531719,0.003

#### 2. Inspect data to see which columns are unneccecary 

In [44]:
# inspect certain columns to see if they can be dropped
print(OMMs_df['ORIGINATOR'].unique())
print(OMMs_df['CENTER_NAME'].unique())
print(OMMs_df['REF_FRAME'].unique())
print(OMMs_df['TIME_SYSTEM'].unique())
print(OMMs_df['MEAN_ELEMENT_THEORY'].unique())
print(OMMs_df['EPHEMERIS_TYPE'].unique())
print(OMMs_df['CLASSIFICATION_TYPE'].unique())
print(OMMs_df['ELEMENT_SET_NO'].unique())

['18 SPCS']
['EARTH']
['TEME']
['UTC']
['SGP4']
['0']
['U']
['999']


In [45]:
# drop not needed columns
OMMs_df.drop(columns=['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'ELEMENT_SET_NO', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2', 'COUNTRY_CODE'], inplace=True)
OMMs_df

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,LAUNCH_DATE,SITE,DECAY_DATE
0,THOR ABLESTAR R/B,1960-003A,1961-08-16T14:24:53.774496,16.29358349,0.00000000,51.2500,356.4892,331.9473,28.0526,30,7590,0.00000000000000,0.02125614,0.0000000000000,6572.405,88.378,194.270,194.270,ROCKET BODY,None,None,None,1961-08-18
1,THOR ABLESTAR DEB,1960-003C,1960-10-06T07:59:06.496512,15.38343354,0.02427570,51.2879,136.0668,196.7800,163.6100,33,2700,0.00000000000000,0.00001932,0.0000000000000,6829.149,93.607,616.796,285.231,DEBRIS,None,None,None,1960-07-17
2,SL-3 R/B,1960-005B,1960-07-16T04:00:11.150208,16.27523673,0.00000000,64.8899,44.6976,36.3819,323.6180,35,990,0.00000000000000,0.03271057,0.0000000000000,6577.344,88.477,199.209,199.209,ROCKET BODY,None,None,None,1960-07-17
3,SPUTNIK 4 DEB,1960-005D,1961-03-15T15:48:42.802271,15.49800338,0.01984030,64.8899,343.2411,333.4201,27.0840,37,4690,0.00000000000000,0.00039007,0.0000000000000,6795.450,92.915,552.139,282.492,DEBRIS,None,None,None,1961-06-30
4,SPUTNIK 4 DEB,1960-005E,1960-08-19T16:34:33.722976,15.88909013,0.00342130,64.8899,302.1218,49.4823,310.6691,38,1500,0.00000000000000,0.00610805,0.0000000000000,6683.481,90.628,328.212,282.480,DEBRIS,None,None,None,1960-08-20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24643,CZ-2F DEB,2024-194B,2024-11-22T23:07:29.851680,15.68300172,0.02578460,41.3237,193.5705,294.4302,62.9836,61684,378,0.00048903000000,0.00465987,0.0000065563000,6741.901,91.819,537.603,189.929,DEBRIS,LARGE,2024-10-29,JSC,2024-12-21
24644,CZ-6A DEB,2024-140ADL,2025-02-15T18:24:08.792352,15.92225175,0.00210800,89.2435,333.2672,98.2724,262.0952,61844,1983,0.04150000000000,0.09622810,0.2057300000000,6674.194,90.439,310.129,281.990,DEBRIS,SMALL,2024-08-06,TSC,2025-02-21
24645,FALCON 9 R/B,2025-022Y,2025-02-19T02:13:45.786720,16.47531719,0.00310630,53.1436,118.6486,5.7660,67.2814,62878,274,0.00011008000000,0.10554311,0.0000130660000,6523.980,87.403,166.111,125.580,ROCKET BODY,LARGE,2025-02-01,AFWTR,2025-02-19
24646,VOLGA R/B,2025-026D,2025-02-05T11:52:04.781856,14.93887108,0.00100970,81.9945,220.5351,276.0393,83.9657,62905,3,0.00656530000000,0.00070488,0.0000000000000,6963.966,96.393,592.863,578.800,ROCKET BODY,None,None,None,2025-02-05


In [46]:
# save to csv
OMMs_df.to_csv("../Data/DS_OMMs.csv", index=False)

In [47]:
# load omms
import pandas as pd
OMMs_df = pd.read_csv("../Data/DS_OMMs.csv")

#### 3. Check the differences between Decay Date and Epoch (timestamp of OMM)

In [ ]:
# convert to date and then again to datetime, needed because otherwise the date conversion does not work :/
OMMs_df['EPOCH'] = pd.to_datetime(OMMs_df['EPOCH']).dt.date
OMMs_df['DECAY_DATE'] = pd.to_datetime(OMMs_df['DECAY_DATE']).dt.date

OMMs_df['EPOCH'] = pd.to_datetime(OMMs_df['EPOCH'])
OMMs_df['DECAY_DATE'] = pd.to_datetime(OMMs_df['DECAY_DATE'])

In [54]:
# check difference between Epoch and Decay Date
OMMs_df['Epoch_Decay_Difference'] = (OMMs_df['DECAY_DATE']-OMMs_df['EPOCH'])
OMMs_df['Epoch_Decay_Difference'].describe()

count                          24648
mean      19 days 03:45:16.650438169
std      300 days 13:11:48.972285292
min            -10727 days +00:00:00
25%                  0 days 00:00:00
50%                  1 days 00:00:00
75%                  5 days 00:00:00
max               8300 days 00:00:00
Name: Epoch_Decay_Difference, dtype: object

In [56]:
# show objects where Epoch_Decay_Difference is negative
OMMs_wrong_df = OMMs_df[OMMs_df['Epoch_Decay_Difference'] < pd.Timedelta(0)]
OMMs_wrong_df

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,LAUNCH_DATE,SITE,DECAY_DATE,Epoch_Decay_Difference
1,THOR ABLESTAR DEB,1960-003C,1960-10-06,15.383434,0.024276,51.2879,136.0668,196.7800,163.6100,33,2700,0.000000,0.000019,0.000000,6829.149,93.607,616.796,285.231,DEBRIS,NaN,NaN,NaN,1960-07-17,-81 days
2872,DELTA 1 DEB,1972-058C,2004-11-24,15.865404,0.001280,98.3597,332.5522,114.4864,245.7741,7832,57786,0.001414,0.003715,0.000000,6690.132,90.763,320.561,303.432,DEBRIS,LARGE,1972-07-23,AFWTR,1977-02-12,-10147 days
3087,COSMOS 777 DEB,1975-102F,1976-05-02,15.506079,0.002667,65.0265,340.8773,6.7572,353.3878,8625,2876,0.003543,0.002125,0.000000,6793.091,92.866,433.072,396.839,DEBRIS,NaN,NaN,NaN,1976-03-15,-48 days
3100,COSMOS 777 DEB,1975-102AE,1976-05-02,15.506415,0.001048,65.0344,341.0843,8.9752,351.1527,8650,2877,0.000000,0.002460,0.000000,6792.993,92.864,421.975,407.741,DEBRIS,NaN,NaN,NaN,1976-03-27,-36 days
3142,COSMOS 777 DEB,1975-102BF,1976-05-02,15.401569,0.005574,64.8281,339.9362,23.9313,336.4347,8728,2872,0.003283,0.001421,0.000000,6823.787,93.496,483.686,407.617,DEBRIS,NaN,NaN,NaN,1976-03-29,-34 days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23254,BION 11 DEB,1996-073G,1997-02-03,16.257886,0.000879,62.7868,87.1680,23.0743,337.0808,24710,439,0.000042,0.001303,0.000009,6582.022,88.572,209.670,198.105,DEBRIS,SMALL,1996-12-24,PKMTR,1997-01-10,-24 days
23897,CZ-2F DEB,2024-078D,2024-04-28,16.315921,0.006350,41.5973,28.4800,152.0982,294.6586,59594,37,0.002455,0.387218,0.000007,6566.402,88.257,229.965,146.568,DEBRIS,NaN,2024-04-25,JSC,2024-04-27,-1 days
24120,COSMOS 2251 DEB,1993-036AZQ,2024-05-11,14.672378,0.003467,73.9910,229.2548,89.3171,271.1971,36387,57450,0.029348,0.001711,0.000000,7048.037,98.144,694.338,645.466,DEBRIS,SMALL,1993-06-16,PKMTR,2024-05-07,-4 days
24312,COSMOS 1408 DEB,1982-092CCH,2023-07-06,15.177755,0.012673,82.5778,267.2573,149.0686,211.8095,57287,1,0.001100,0.000275,0.000000,6890.702,94.876,599.895,425.239,DEBRIS,NaN,NaN,NaN,2023-06-12,-24 days


In [57]:
# save IDs of these objects to csv to later also delete from SATCAT data
wrong_ids = OMMs_wrong_df['NORAD_CAT_ID']
wrong_ids.to_csv("../Data/DS_WrongIDs.csv", index=False)

In [58]:
# drop objects where OMM recorded after object was supposed to have decayed
OMMs_df = OMMs_df[OMMs_df['Epoch_Decay_Difference'] >= pd.Timedelta(0)]
print(f"Remaining objects: {len(OMMs_df)}")



Remaining objects: 24222


In [59]:
# Desribe in days
OMMs_df['Epoch_Decay_Difference'] = OMMs_df['Epoch_Decay_Difference'].dt.days
OMMs_df['Epoch_Decay_Difference'].describe()

count    24222.000000
mean        34.451160
std        222.983892
min          0.000000
25%          0.000000
50%          1.000000
75%          5.000000
max       8300.000000
Name: Epoch_Decay_Difference, dtype: float64

In [60]:
# remove the difference and epoch column 
OMMs_df.drop(columns=['EPOCH', 'Epoch_Decay_Difference'], inplace=True)

In [61]:
# Save OMMs to csv
OMMs_df.to_csv('../Data/DS_OMMs.csv', index=False)